# PCL detection: class weight schedule experiments

RoBERTa-base with HTML stripping. **Experiment:** Compare 3 class weight strategies:
1. **Fixed**: Constant class weights throughout training
2. **Increasing**: Start with lower weights for minority class, linearly increase to final weights
3. **Decreasing**: Start with higher weights for minority class, linearly decrease to final weights

In [1]:
import html
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Resolve project root regardless of whether notebook is run from project root or src/
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

/vol/bitbucket/tq422/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Project root: /vol/bitbucket/tq422/NLP_cw


## Load data and build train/dev splits

In [2]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

# Split IDs (match organizer: exactly one row per ID from split files)
a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

train_df = a_train[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"] = dev_df["text"].fillna("").astype(str)
train_df["label_binary"] = train_df["label_binary"].fillna(0).astype(int)
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Split a validation set from training data; dev is held out and not used during training
VAL_RATIO = 0.1
train_df, val_df = train_test_split(
    train_df, test_size=VAL_RATIO, stratify=train_df["label_binary"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Dev size:   {len(dev_df)} (held out, not used during training)")
print("\nTrain label distribution:")
print(train_df["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Train size: 7537
Val size:   838
Dev size:   2094 (held out, not used during training)

Train label distribution:
label_binary
0    6822
1     715
Name: count, dtype: int64

Val label distribution:
label_binary
0    759
1     79
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


## Preprocess text: remove HTML markups

In [3]:
def strip_html_and_clean(text: str) -> str:
    """Remove HTML tags, unescape entities, and normalize whitespace."""
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    # Remove HTML tags (pattern from data_exploration: <[^>]+>)
    no_tags = re.sub(r"<[^>]+>", "", text)
    # Unescape HTML entities (e.g. &amp; -> &, &lt; -> <)
    unescaped = html.unescape(no_tags)
    # Collapse multiple spaces and strip
    return re.sub(r"\s+", " ", unescaped).strip()


train_df["text"] = train_df["text"].apply(strip_html_and_clean)
val_df["text"] = val_df["text"].apply(strip_html_and_clean)
dev_df["text"] = dev_df["text"].apply(strip_html_and_clean)

# Sanity check: ensure no empty texts
train_df["text"] = train_df["text"].replace("", " ")
val_df["text"] = val_df["text"].replace("", " ")
dev_df["text"] = dev_df["text"].replace("", " ")
print("HTML preprocessing applied to train, val, and dev texts.")

HTML preprocessing applied to train, val, and dev texts.


## Compute base class weights

In [4]:
# Compute base class weights (inverse frequency)
n0 = (train_df["label_binary"] == 0).sum()
n1 = (train_df["label_binary"] == 1).sum()
n = n0 + n1
w0_base = n / (2 * n0)
w1_base = n / (2 * n1)
base_class_weights = torch.tensor([w0_base, w1_base], dtype=torch.float32)

print(f"Base class weights (0, 1): {base_class_weights.tolist()}")
print(f"Weight ratio (w1/w0): {w1_base / w0_base:.4f}")

Base class weights (0, 1): [0.55240398645401, 5.270629405975342]
Weight ratio (w1/w0): 9.5413


## Prepare datasets and tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(train_df["text"], train_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Define Trainer with dynamic class weights

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }


class DynamicWeightTrainer(Trainer):
    """Trainer with dynamic class weights that can change per epoch."""

    def __init__(self, weight_schedule="fixed", base_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.weight_schedule = weight_schedule  # 'fixed', 'increasing', 'decreasing'
        self.base_weights = base_weights  # Base weights [w0, w1]
        self.num_epochs = kwargs.get("args").num_train_epochs if kwargs.get("args") else EPOCHS
        
        # For increasing: start at 0.5x base weight for minority class, end at base weight
        # For decreasing: start at 1.5x base weight for minority class, end at base weight
        if base_weights is not None:
            w0_base, w1_base = base_weights[0].item(), base_weights[1].item()
            if weight_schedule == "increasing":
                self.w1_start = w1_base * 0.5  # Start at 50% of base
                self.w1_end = w1_base
            elif weight_schedule == "decreasing":
                self.w1_start = w1_base * 1.5  # Start at 150% of base
                self.w1_end = w1_base
            else:  # fixed
                self.w1_start = w1_base
                self.w1_end = w1_base
        else:
            self.w1_start = None
            self.w1_end = None

    def get_current_weights(self):
        """Get class weights for current epoch based on schedule."""
        if self.base_weights is None:
            return None
        
        if self.weight_schedule == "fixed":
            return self.base_weights
        
        # Get current epoch (0-indexed, so epoch 1 is 0.0, epoch 2 is 1.0, etc.)
        # Use state.epoch which is 1-indexed, so we subtract 1
        if hasattr(self.state, "epoch") and self.state.epoch is not None:
            current_epoch = self.state.epoch - 1  # Convert to 0-indexed
        else:
            current_epoch = 0
        
        # Clamp to valid range
        current_epoch = max(0, min(current_epoch, self.num_epochs - 1))
        
        # Linear interpolation
        if self.num_epochs > 1:
            progress = current_epoch / (self.num_epochs - 1)
        else:
            progress = 0.0
        
        w0 = self.base_weights[0].item()
        w1 = self.w1_start + progress * (self.w1_end - self.w1_start)
        
        return torch.tensor([w0, w1], dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Get current weights based on schedule
        current_weights = self.get_current_weights()
        
        if current_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=current_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss


print("DynamicWeightTrainer class defined.")

DynamicWeightTrainer class defined.


In [7]:
print("=" * 60)
print("EXPERIMENT 1: Fixed Class Weights")
print("=" * 60)

MODEL_DIR_1 = PROJECT_ROOT / "models" / "roberta_pcl_weight_fixed"
MODEL_DIR_1.mkdir(parents=True, exist_ok=True)

model_1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_1 = TrainingArguments(
    output_dir=str(MODEL_DIR_1 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_1 = DynamicWeightTrainer(
    model=model_1,
    args=training_args_1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    weight_schedule="fixed",
    base_weights=base_class_weights,
)

print(f"Fixed weights: {base_class_weights.tolist()}")
train_result_1 = trainer_1.train()
print("Training finished.")

EXPERIMENT 1: Fixed Class Weights


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fixed weights: [0.55240398645401, 5.270629405975342]


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.516600,0.646809,0.583532,0.178571,0.949367,0.300601,0.502042
2,0.447900,0.448456,0.860382,0.378205,0.746835,0.502128,0.710467
3,0.331800,0.886459,0.912888,0.541667,0.493671,0.516556,0.734344
4,0.208800,1.055314,0.899761,0.476190,0.632911,0.543478,0.743589
5,0.053800,1.826656,0.912888,0.553571,0.392405,0.459259,0.705944
6,0.111500,1.839976,0.923628,0.636364,0.443038,0.522388,0.740442
7,0.018600,1.849940,0.917661,0.571429,0.506329,0.536913,0.745863


Training finished.


In [8]:
# Evaluate on dev set
eval_metrics_1 = trainer_1.evaluate(dev_dataset)
print("\nDev metrics (Fixed):")
for k, v in eval_metrics_1.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")

pred_output_1 = trainer_1.predict(dev_dataset)
dev_preds_1 = np.argmax(pred_output_1.predictions, axis=-1)
dev_labels = pred_output_1.label_ids

report_str_1 = classification_report(dev_labels, dev_preds_1, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (dev, Fixed):")
print(report_str_1)

dev_metrics_record_1 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_1.items()}
dev_metrics_record_1["classification_report"] = report_str_1
dev_metrics_record_1["weight_schedule"] = "fixed"
dev_metrics_record_1["final_weights"] = base_class_weights.tolist()
dev_metrics_path_1 = OUTPUT_DIR / "roberta_pcl_weight_fixed_dev_metrics.json"
with open(dev_metrics_path_1, "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_1, f, indent=2)
print(f"\nSaved dev metrics to: {dev_metrics_path_1}")

# Save model
trainer_1.save_model(str(MODEL_DIR_1))
tokenizer.save_pretrained(str(MODEL_DIR_1))


Dev metrics (Fixed):
eval_loss: 0.4006
eval_accuracy: 0.9140
eval_precision_pos: 0.5391
eval_recall_pos: 0.6583
eval_f1_pos: 0.5928
eval_f1_macro: 0.7724
eval_runtime: 7.0295
eval_samples_per_second: 297.8880
eval_steps_per_second: 9.3890
epoch: 7.0000

Classification report (dev, Fixed):
              precision    recall  f1-score   support

      No PCL     0.9633    0.9409    0.9519      1895
         PCL     0.5391    0.6583    0.5928       199

    accuracy                         0.9140      2094
   macro avg     0.7512    0.7996    0.7724      2094
weighted avg     0.9230    0.9140    0.9178      2094


Saved dev metrics to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_weight_fixed_dev_metrics.json


('/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/tokenizer_config.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/special_tokens_map.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/vocab.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/merges.txt',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/added_tokens.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_fixed/tokenizer.json')

## Experiment 2: Increasing class weights

In [9]:
print("=" * 60)
print("EXPERIMENT 2: Increasing Class Weights")
print("=" * 60)

MODEL_DIR_2 = PROJECT_ROOT / "models" / "roberta_pcl_weight_increasing"
MODEL_DIR_2.mkdir(parents=True, exist_ok=True)

model_2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_2 = TrainingArguments(
    output_dir=str(MODEL_DIR_2 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_2 = DynamicWeightTrainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    weight_schedule="increasing",
    base_weights=base_class_weights,
)

w1_start = base_class_weights[1].item() * 0.5
w1_end = base_class_weights[1].item()
print(f"Weight schedule: w1 starts at {w1_start:.4f}, ends at {w1_end:.4f}")
print(f"(w0 remains constant at {base_class_weights[0].item():.4f})")
train_result_2 = trainer_2.train()
print("Training finished.")

EXPERIMENT 2: Increasing Class Weights


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Weight schedule: w1 starts at 2.6353, ends at 5.2706
(w0 remains constant at 0.5524)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.451700,0.579390,0.670644,0.207715,0.886076,0.336538,0.558745
2,0.377400,0.469933,0.900955,0.480392,0.620253,0.541436,0.742959
3,0.285600,0.689149,0.908115,0.511905,0.544304,0.527607,0.738358
4,0.168200,0.924572,0.896181,0.462264,0.620253,0.529730,0.735690
5,0.044300,1.428978,0.912888,0.544118,0.468354,0.503401,0.727829


Training finished.


In [10]:
# Evaluate on dev set
eval_metrics_2 = trainer_2.evaluate(dev_dataset)
print("\nDev metrics (Increasing):")
for k, v in eval_metrics_2.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")

pred_output_2 = trainer_2.predict(dev_dataset)
dev_preds_2 = np.argmax(pred_output_2.predictions, axis=-1)

report_str_2 = classification_report(dev_labels, dev_preds_2, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (dev, Increasing):")
print(report_str_2)

dev_metrics_record_2 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_2.items()}
dev_metrics_record_2["classification_report"] = report_str_2
dev_metrics_record_2["weight_schedule"] = "increasing"
w1_start = base_class_weights[1].item() * 0.5
dev_metrics_record_2["weight_start"] = [base_class_weights[0].item(), w1_start]
dev_metrics_record_2["weight_end"] = base_class_weights.tolist()
dev_metrics_path_2 = OUTPUT_DIR / "roberta_pcl_weight_increasing_dev_metrics.json"
with open(dev_metrics_path_2, "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_2, f, indent=2)
print(f"\nSaved dev metrics to: {dev_metrics_path_2}")

# Save model
trainer_2.save_model(str(MODEL_DIR_2))
tokenizer.save_pretrained(str(MODEL_DIR_2))


Dev metrics (Increasing):
eval_loss: 0.2248
eval_accuracy: 0.9040
eval_precision_pos: 0.4961
eval_recall_pos: 0.6382
eval_f1_pos: 0.5582
eval_f1_macro: 0.7522
eval_runtime: 7.0697
eval_samples_per_second: 296.1940
eval_steps_per_second: 9.3360
epoch: 5.0000

Classification report (dev, Increasing):
              precision    recall  f1-score   support

      No PCL     0.9608    0.9319    0.9462      1895
         PCL     0.4961    0.6382    0.5582       199

    accuracy                         0.9040      2094
   macro avg     0.7285    0.7851    0.7522      2094
weighted avg     0.9167    0.9040    0.9093      2094


Saved dev metrics to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_weight_increasing_dev_metrics.json


('/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/tokenizer_config.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/special_tokens_map.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/vocab.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/merges.txt',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/added_tokens.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_increasing/tokenizer.json')

## Experiment 3: Decreasing class weights

In [11]:
print("=" * 60)
print("EXPERIMENT 3: Decreasing Class Weights")
print("=" * 60)

MODEL_DIR_3 = PROJECT_ROOT / "models" / "roberta_pcl_weight_decreasing"
MODEL_DIR_3.mkdir(parents=True, exist_ok=True)

model_3 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args_3 = TrainingArguments(
    output_dir=str(MODEL_DIR_3 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_3 = DynamicWeightTrainer(
    model=model_3,
    args=training_args_3,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    weight_schedule="decreasing",
    base_weights=base_class_weights,
)

w1_start = base_class_weights[1].item() * 1.5
w1_end = base_class_weights[1].item()
print(f"Weight schedule: w1 starts at {w1_start:.4f}, ends at {w1_end:.4f}")
print(f"(w0 remains constant at {base_class_weights[0].item():.4f})")
train_result_3 = trainer_3.train()
print("Training finished.")

EXPERIMENT 3: Decreasing Class Weights


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Weight schedule: w1 starts at 7.9059, ends at 5.2706
(w0 remains constant at 0.5524)


Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.505600,0.557080,0.597852,0.183824,0.949367,0.308008,0.512288
2,0.444600,0.480491,0.849642,0.360947,0.772152,0.491935,0.701850
3,0.408000,0.985029,0.911695,0.529412,0.569620,0.548780,0.749919
4,0.201700,1.255788,0.908115,0.510417,0.620253,0.560000,0.754350
5,0.063400,2.419941,0.911695,0.549020,0.354430,0.430769,0.691452
6,0.113900,1.687383,0.900955,0.478261,0.556962,0.514620,0.729735
7,0.032700,2.219809,0.915274,0.562500,0.455696,0.503497,0.728591


Training finished.


In [12]:
# Evaluate on dev set
eval_metrics_3 = trainer_3.evaluate(dev_dataset)
print("\nDev metrics (Decreasing):")
for k, v in eval_metrics_3.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")

pred_output_3 = trainer_3.predict(dev_dataset)
dev_preds_3 = np.argmax(pred_output_3.predictions, axis=-1)

report_str_3 = classification_report(dev_labels, dev_preds_3, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (dev, Decreasing):")
print(report_str_3)

dev_metrics_record_3 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_3.items()}
dev_metrics_record_3["classification_report"] = report_str_3
dev_metrics_record_3["weight_schedule"] = "decreasing"
w1_start = base_class_weights[1].item() * 1.5
dev_metrics_record_3["weight_start"] = [base_class_weights[0].item(), w1_start]
dev_metrics_record_3["weight_end"] = base_class_weights.tolist()
dev_metrics_path_3 = OUTPUT_DIR / "roberta_pcl_weight_decreasing_dev_metrics.json"
with open(dev_metrics_path_3, "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_3, f, indent=2)
print(f"\nSaved dev metrics to: {dev_metrics_path_3}")

# Save model
trainer_3.save_model(str(MODEL_DIR_3))
tokenizer.save_pretrained(str(MODEL_DIR_3))


Dev metrics (Decreasing):
eval_loss: 0.3553
eval_accuracy: 0.9164
eval_precision_pos: 0.5508
eval_recall_pos: 0.6533
eval_f1_pos: 0.5977
eval_f1_macro: 0.7755
eval_runtime: 7.0157
eval_samples_per_second: 298.4750
eval_steps_per_second: 9.4080
epoch: 7.0000

Classification report (dev, Decreasing):
              precision    recall  f1-score   support

      No PCL     0.9629    0.9441    0.9534      1895
         PCL     0.5508    0.6533    0.5977       199

    accuracy                         0.9164      2094
   macro avg     0.7569    0.7987    0.7755      2094
weighted avg     0.9237    0.9164    0.9196      2094


Saved dev metrics to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_weight_decreasing_dev_metrics.json


('/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/tokenizer_config.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/special_tokens_map.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/vocab.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/merges.txt',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/added_tokens.json',
 '/vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_weight_decreasing/tokenizer.json')

## Summary: Compare all experiments

In [ ]:
print("=" * 60)
print("SUMMARY: Comparison of All Experiments")
print("=" * 60)

results_summary = {
    "Fixed": {
        "f1_pos": eval_metrics_1.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_1.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_1.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_1.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_1.get("eval_accuracy", 0),
    },
    "Increasing": {
        "f1_pos": eval_metrics_2.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_2.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_2.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_2.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_2.get("eval_accuracy", 0),
    },
    "Decreasing": {
        "f1_pos": eval_metrics_3.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_3.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_3.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_3.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_3.get("eval_accuracy", 0),
    },
}

print("\nDev Set Performance:")
print(f"{'Metric':<20} {'Fixed':<12} {'Increasing':<12} {'Decreasing':<12}")
print("-" * 60)
for metric in ["f1_pos", "f1_macro", "precision_pos", "recall_pos", "accuracy"]:
    print(f"{metric:<20} {results_summary['Fixed'][metric]:<12.4f} {results_summary['Increasing'][metric]:<12.4f} {results_summary['Decreasing'][metric]:<12.4f}")

# Save summary
summary_path = OUTPUT_DIR / "roberta_pcl_weight_schedule_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2)
print(f"\nSaved summary to: {summary_path}")

# Find best performing
best_f1_pos = max(results_summary.items(), key=lambda x: x[1]["f1_pos"])
print(f"\nBest F1-Pos: {best_f1_pos[0]} ({best_f1_pos[1]['f1_pos']:.4f})")
best_f1_macro = max(results_summary.items(), key=lambda x: x[1]["f1_macro"])
print(f"Best F1-Macro: {best_f1_macro[0]} ({best_f1_macro[1]['f1_macro']:.4f})")

SUMMARY: Comparison of All Experiments

Dev Set Performance:
Metric               Fixed        Increasing   Decreasing  
------------------------------------------------------------
f1_pos               0.5928       0.5582       0.5977      
f1_macro             0.7724       0.7522       0.7755      
precision_pos        0.5391       0.4961       0.5508      
recall_pos           0.6583       0.6382       0.6533      
accuracy             0.9140       0.9040       0.9164      

Saved summary to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_weight_schedule_summary.json

Best F1-Pos: Decreasing (0.5977)
Best F1-Macro: Decreasing (0.7755)


: 